In [ ]:
from google.colab import files
uploaded = files.upload()


Saving Practice_data_with_pass_fail.xlsx to Practice_data_with_pass_fail.xlsx


In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, classification_report
)
df = pd.read_excel(next(iter(uploaded)))
print(df.head())

print("\nShape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing values:\n", df.isnull().sum())


   Student_ID Department  Gender  Attendance_Pct  Study_Hours_Per_Day  \
0        1001        CSE    Male            82.0                  7.5   
1        1002         IT  Female            91.0                  8.2   
2        1003        ECE    Male            76.0                  5.5   
3        1004      AI&DS  Female            88.0                  7.0   
4        1005        BBA  Female            72.0                  4.0   

   Internal_Marks  Assignments_Completed  Final_Score pass_fail  
0            68.0                    9.0           78      Pass  
1            74.0                   10.0           86      Pass  
2            61.0                    7.0           70      Pass  
3            79.0                    9.0           84      Pass  
4            58.0                    6.0           64      Fail  

Shape: (28, 9)

Columns: ['Student_ID', 'Department', 'Gender', 'Attendance_Pct', 'Study_Hours_Per_Day', 'Internal_Marks', 'Assignments_Completed', 'Final_Score', '

In [ ]:

# Input features
X = df.drop(
    columns=["Student_ID", "Final_Score", "pass_fail"]
)

# Convert Department and Gender into numeric columns
X = pd.get_dummies(X, drop_first=True, dtype=int)

# Target for classification
y = df["pass_fail"]

print("Input shape:", X.shape)
print("\nTarget values:\n", y.value_counts())


Input shape: (28, 9)

Target values:
 pass_fail
Pass    20
Fail     8
Name: count, dtype: int64


In [ ]:

x_train, x_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training records:", len(x_train))
print("Testing records:", len(x_test))


Training records: 22
Testing records: 6


In [ ]:

models = {
    "Logistic Regression": make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=1000)
    ),

    "Decision Tree": make_pipeline(
        SimpleImputer(strategy="median"),
        DecisionTreeClassifier(
            max_depth=4, random_state=42
        )
    ),

    "Random Forest": make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(
            n_estimators=100, random_state=42
        )
    )
}

for name, model in models.items():
    model.fit(x_train, y_train)
    prediction = model.predict(x_test)

    print(f"\n{'=' * 35}")
    print(name)
    print("Accuracy:", round(
        accuracy_score(y_test, prediction), 3
    ))
    print(classification_report(
        y_test, prediction, zero_division=0
    ))



Logistic Regression
Accuracy: 0.667
              precision    recall  f1-score   support

        Fail       0.00      0.00      0.00         2
        Pass       0.67      1.00      0.80         4

    accuracy                           0.67         6
   macro avg       0.33      0.50      0.40         6
weighted avg       0.44      0.67      0.53         6


Decision Tree
Accuracy: 0.667
              precision    recall  f1-score   support

        Fail       0.50      0.50      0.50         2
        Pass       0.75      0.75      0.75         4

    accuracy                           0.67         6
   macro avg       0.62      0.62      0.62         6
weighted avg       0.67      0.67      0.67         6


Random Forest
Accuracy: 0.833
              precision    recall  f1-score   support

        Fail       1.00      0.50      0.67         2
        Pass       0.80      1.00      0.89         4

    accuracy                           0.83         6
   macro avg       0.90      

In [ ]:

# Numerical target
y_score = df["Final_Score"]

# Use the same input features
X_score = X.copy()

# Split data
x_train_lr, x_test_lr, y_train_lr, y_test_lr = (
    train_test_split(
        X_score, y_score,
        test_size=0.2,
        random_state=42
    )
)

# Create and train the model
linear_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LinearRegression()
)

linear_model.fit(x_train_lr, y_train_lr)

# Predict scores
score_prediction = linear_model.predict(x_test_lr)

print("Actual scores:", y_test_lr.tolist())
print("Predicted scores:", np.round(score_prediction, 2))

print("\nMAE:", round(
    mean_absolute_error(y_test_lr, score_prediction), 2
))
print("RMSE:", round(
    np.sqrt(mean_squared_error(y_test_lr, score_prediction)), 2
))
print("R2 Score:", round(
    r2_score(y_test_lr, score_prediction), 3
))


Actual scores: [90, 94, 73, 66, 78, 69]
Predicted scores: [91.35 94.01 75.2  64.85 80.73 68.03]

MAE: 1.4
RMSE: 1.65
R2 Score: 0.975
